##CAMADA BRONZE

Leitura dos CSVs semestrais da ANP (2016 a 2026)

Nessa camada os dados ficaram da forma que chegaram, como texto. Sendo apenas renomeado as colunas.

In [0]:
from pyspark.sql import functions as F
import os
 
caminho = "/Volumes/workspace/default/dados_anp"
 
spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.bronze")
spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.silver")
spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.gold")

##Verificação se todos os arquivos estão corretos

In [0]:
arquivos = sorted(f for f in os.listdir(caminho) if f.lower().endswith(".csv"))
 
print(len(arquivos), "arquivos")
for a in arquivos:
    tamanho = os.path.getsize(f"{caminho}/{a}") / 1024 / 1024
    print(f"{a}  ({tamanho:.1f} MB)")

##Verificação se todos estão com as mesmas colunas

In [0]:
cabecalhos = {}
for a in arquivos:
    with open(f"{caminho}/{a}", encoding="utf-8-sig", errors="replace") as f:
        cabecalhos[a] = f.readline().strip()
 
distintos = set(cabecalhos.values())
print("cabeçalhos diferentes:", len(distintos))
 
for c in distintos:
    qtd = sum(1 for v in cabecalhos.values() if v == c)
    print(f"\n{qtd} arquivos com:\n{c}")

##Leitura CSV

In [0]:
df = (
    spark.read
    .option("header", True)
    .option("sep", ";")
    .option("encoding", "UTF-8")
    .csv(f"{caminho}/*.csv")
    .withColumn("arquivo_origem", F.col("_metadata.file_name"))
)
 
df.printSchema()

##Renomear as Colunas
Trocar os nomes originais para melhor organização

In [0]:
nomes = {
    "Regiao - Sigla": "regiao_sigla",
    "Estado - Sigla": "estado_sigla",
    "Municipio": "municipio",
    "Revenda": "revenda",
    "CNPJ da Revenda": "cnpj_revenda",
    "Nome da Rua": "nome_rua",
    "Numero Rua": "numero_rua",
    "Complemento": "complemento",
    "Bairro": "bairro",
    "Cep": "cep",
    "Produto": "produto",
    "Data da Coleta": "data_coleta",
    "Valor de Venda": "valor_venda",
    "Valor de Compra": "valor_compra",
    "Unidade de Medida": "unidade_medida",
    "Bandeira": "bandeira",
}
 
for coluna in df.columns:
    limpa = coluna.replace("", "").strip()
    if limpa in nomes:
        df = df.withColumnRenamed(coluna, nomes[limpa])
 
df = df.withColumn("data_ingestao", F.current_timestamp())
 
print(df.columns)

##Gravando a tabela bronze

In [0]:

(
    df.write
    .mode("overwrite")
    .option("overwriteSchema", True)
    .saveAsTable("workspace.bronze.precos_combustiveis")
)
 
bronze = spark.table("workspace.bronze.precos_combustiveis")
print("registros:", bronze.count())

In [0]:
display(
    bronze.groupBy("arquivo_origem").count().orderBy("arquivo_origem")
)
 

## Checar

In [0]:
display(bronze.limit(10))

## Descrição no catálogo

In [0]:

spark.sql("""
COMMENT ON TABLE workspace.bronze.precos_combustiveis IS
'Dados brutos da Serie Historica de Precos de Combustiveis Automotivos da ANP, 2016 a 2026.
Uniao dos CSVs semestrais sem alteracao de valores. Todas as colunas em texto.
Uma linha por coleta de preco (posto x produto x data).'
""")
 
descricoes = {
    "regiao_sigla": "Sigla da regiao (N, NE, CO, SE, S)",
    "estado_sigla": "Sigla da UF",
    "municipio": "Municipio do posto",
    "revenda": "Razao social do posto",
    "cnpj_revenda": "CNPJ do posto, formatado e com espaco a esquerda na origem",
    "nome_rua": "Logradouro",
    "numero_rua": "Numero do endereco (texto, aceita S/N)",
    "complemento": "Complemento do endereco",
    "bairro": "Bairro",
    "cep": "CEP",
    "produto": "Combustivel: GASOLINA, GASOLINA ADITIVADA, ETANOL, DIESEL, DIESEL S10, GNV",
    "data_coleta": "Data da coleta em texto dd/MM/yyyy",
    "valor_venda": "Preco de venda em texto com virgula decimal",
    "valor_compra": "Preco de compra do posto. ANP parou de publicar em ago/2020",
    "unidade_medida": "R$ / litro ou R$ / m3 (GNV)",
    "bandeira": "Bandeira do posto. BRANCA = sem bandeira",
    "arquivo_origem": "Arquivo CSV de onde veio a linha",
    "data_ingestao": "Data e hora da carga",
}
 
for col, desc in descricoes.items():
    spark.sql(f"ALTER TABLE workspace.bronze.precos_combustiveis ALTER COLUMN {col} COMMENT '{desc}'")
 